# CircleSense: Overlapping Friend-Circle Detection with Clique Percolation + GNNs
**BDSMA micro-project (Sem 7)** · Based on *Paper 4: Overlapping Community Detection: Combining CPM with GNNs*

This notebook runs the **entire experiment end to end**, with no repository needed:

1. Install dependencies and download SNAP `ego-Facebook`
2. Write the CircleSense source package (seeding, GNN, baselines, metrics, analysis)
3. Walk through the pipeline on one ego-network (seeds → GNN → circles)
4. Batch run on all 10 ego-networks and compare **Hybrid vs CPM vs Louvain vs BigCLAM vs GNN-without-seeds**
5. Bridge friends, friend suggestions and circle explanations
6. Visualisations
7. *(Optional)* ablation / tuning sweep

**Runtime:** about 1–3 min on CPU for the main sections; the full ablation takes ~15 min (a quick mode is provided). A GPU is not required.

## 1. Setup

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "networkx", "numpy", "scikit-learn", "scipy", "torch", "matplotlib", "pandas"], check=False)
import torch, networkx, numpy, sklearn
print("torch", torch.__version__, "| networkx", networkx.__version__, "| numpy", numpy.__version__)

### 1.1 Download the dataset (SNAP `ego-Facebook`: 10 ego-networks, 4,039 nodes, 88,234 edges)

In [ ]:
import os, tarfile, urllib.request

WORK = "/content/circlesense" if os.path.isdir("/content") else os.path.abspath("circlesense")
os.makedirs(f"{WORK}/data/raw", exist_ok=True)
for d in ["app", "app/data", "app/seeding", "app/model", "app/baselines", "app/evaluation", "app/analysis", "scripts", "experiments"]:
    os.makedirs(f"{WORK}/backend/{d}", exist_ok=True)
os.chdir(WORK)

URL = "https://snap.stanford.edu/data/facebook.tar.gz"
TAR = "data/raw/facebook.tar.gz"
if not os.path.isdir("data/raw/facebook"):
    try:
        urllib.request.urlretrieve(URL, TAR)
        with tarfile.open(TAR) as t:
            t.extractall("data/raw")
    except Exception as e:
        print("Download failed:", e)
        print("Upload facebook.tar.gz manually in the dialog below.")
        from google.colab import files
        up = files.upload()
        os.replace(list(up)[0], TAR)
        with tarfile.open(TAR) as t:
            t.extractall("data/raw")
print(sorted(f for f in os.listdir("data/raw/facebook") if f.endswith(".edges")))

## 2. Source code
The cells below write the CircleSense package (`backend/app/...`) to disk with `%%writefile`, identical to the project repository.
Expand any cell to read that module.

**`backend/app/__init__.py`**: Package

In [ ]:
%%writefile backend/app/__init__.py
# BDSMA Microproject backend package

**`backend/app/data/__init__.py`**: Data package

In [ ]:
%%writefile backend/app/data/__init__.py
# Data loader package
from .loader import list_egos, load_ego, to_graph_tensors

__all__ = ["list_egos", "load_ego", "to_graph_tensors"]

**`backend/app/data/loader.py`**: Loader: reads SNAP edges, features, circles; builds dense tensors

In [ ]:
%%writefile backend/app/data/loader.py
import os
import glob
from typing import Dict, List, Tuple, Optional
import networkx as nx
import numpy as np
import torch


def get_data_dir() -> str:
    """Find the path to data/raw/facebook robustly."""
    curr = os.path.abspath(os.path.dirname(__file__))
    candidates = [
        os.path.join(curr, "..", "..", "..", "data", "raw", "facebook"),
        os.path.join(curr, "..", "..", "data", "raw", "facebook"),
        os.path.join(os.getcwd(), "data", "raw", "facebook"),
        os.path.join(os.getcwd(), "..", "data", "raw", "facebook"),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return os.path.abspath(c)
    # Default fallback
    return os.path.abspath(candidates[0])


def list_egos(data_dir: Optional[str] = None) -> List[int]:
    """Return sorted list of available ego IDs."""
    if data_dir is None:
        data_dir = get_data_dir()
    edge_files = glob.glob(os.path.join(data_dir, "*.edges"))
    ego_ids = []
    for f in edge_files:
        base = os.path.splitext(os.path.basename(f))[0]
        if base.isdigit():
            ego_ids.append(int(base))
    return sorted(ego_ids)


def load_ego(ego_id: int, data_dir: Optional[str] = None):
    """
    Load network, profile features, and circles for an ego-network.

    Returns:
        G (nx.Graph): NetworkX graph with node IDs as strings, with node attribute 'index'
        node_list (List[str]): List of all friend IDs in order 0..N-1
        node_to_idx (Dict[str, int]): Mapping from node ID string to index 0..N-1
        X (np.ndarray): Shape (N, D), binary feature matrix
        circles (Dict[str, List[str]]): Ground-truth circles mapping name -> list of node IDs
        feat_names (List[str]): List of feature description strings
    """
    if data_dir is None:
        data_dir = get_data_dir()

    base_path = os.path.join(data_dir, str(ego_id))
    feat_path = f"{base_path}.feat"
    edges_path = f"{base_path}.edges"
    circles_path = f"{base_path}.circles"
    featnames_path = f"{base_path}.featnames"

    if not os.path.exists(feat_path):
        raise FileNotFoundError(f"Missing feature file: {feat_path}")

    # 1. Read features and collect all friend nodes
    node_features: Dict[str, np.ndarray] = {}
    with open(feat_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            node_id = parts[0]
            feats = np.array([int(x) for x in parts[1:]], dtype=np.float32)
            node_features[node_id] = feats

    node_list = sorted(list(node_features.keys()), key=lambda x: int(x) if x.isdigit() else x)
    node_to_idx = {nid: i for i, nid in enumerate(node_list)}
    N = len(node_list)
    D = len(next(iter(node_features.values()))) if N > 0 else 0

    X = np.zeros((N, D), dtype=np.float32)
    for i, nid in enumerate(node_list):
        X[i] = node_features[nid]

    # 2. Read graph edges
    G = nx.Graph()
    for nid in node_list:
        G.add_node(nid, index=node_to_idx[nid])

    if os.path.exists(edges_path):
        with open(edges_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    u, v = parts[0], parts[1]
                    if u in node_to_idx and v in node_to_idx:
                        G.add_edge(u, v)

    # 3. Read ground truth circles
    circles: Dict[str, List[str]] = {}
    if os.path.exists(circles_path):
        with open(circles_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if parts:
                    cname = parts[0]
                    members = [p for p in parts[1:] if p in node_to_idx]
                    circles[cname] = members

    # 4. Read feature names if available
    feat_names: List[str] = []
    if os.path.exists(featnames_path):
        with open(featnames_path, "r", encoding="utf-8") as f:
            for line in f:
                feat_names.append(line.strip())

    return G, node_list, node_to_idx, X, circles, feat_names


def to_graph_tensors(G: nx.Graph, X: np.ndarray, S: Optional[np.ndarray] = None):
    """
    Convert graph and features to PyTorch tensors.

    Returns:
        edge_index (torch.LongTensor): Shape (2, 2*E)
        adj_dense (torch.FloatTensor): Shape (N, N)
        adj_norm (torch.FloatTensor): Normalized adjacency (N, N) for SAGE/GCN
        X_tensor (torch.FloatTensor): Shape (N, D)
        S_tensor (torch.FloatTensor or None): Shape (N, K)
    """
    N = len(G.nodes)
    adj_dense = np.zeros((N, N), dtype=np.float32)
    edges = []

    for u, v in G.edges():
        u_idx = G.nodes[u]["index"]
        v_idx = G.nodes[v]["index"]
        adj_dense[u_idx, v_idx] = 1.0
        adj_dense[v_idx, u_idx] = 1.0
        edges.append((u_idx, v_idx))
        edges.append((v_idx, u_idx))

    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    # Normalized adjacency with self-loops: D^{-1/2} (A + I) D^{-1/2}
    A_self = adj_dense + np.eye(N, dtype=np.float32)
    deg = np.sum(A_self, axis=1)
    deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
    deg_inv_sqrt[deg <= 0] = 0.0
    deg_mat = np.diag(deg_inv_sqrt)
    adj_norm = deg_mat @ A_self @ deg_mat

    adj_dense_t = torch.from_numpy(adj_dense)
    adj_norm_t = torch.from_numpy(adj_norm.astype(np.float32))
    X_t = torch.from_numpy(X)
    S_t = torch.from_numpy(S.astype(np.float32)) if S is not None else None

    return edge_index, adj_dense_t, adj_norm_t, X_t, S_t

**`backend/app/seeding/__init__.py`**: Seeding package

In [ ]:
%%writefile backend/app/seeding/__init__.py
# Seeding package for Clique Percolation and Weak Cliques
from .cpm import k_clique_seeds
from .weak_clique import weak_clique_seeds
from .merge import merge_seeds, build_seed_matrix

__all__ = ["k_clique_seeds", "weak_clique_seeds", "merge_seeds", "build_seed_matrix"]

**`backend/app/seeding/cpm.py`**: k-clique percolation seeds

In [ ]:
%%writefile backend/app/seeding/cpm.py
from typing import List, Set, Dict, Any
import networkx as nx


def k_clique_seeds(G: nx.Graph, k: int = 4, max_cliques: int = 500) -> List[Dict[str, Any]]:
    """
    Find clique percolation communities in graph G.
    To avoid combinatorial explosion on dense graphs (where max clique size > 15),
    we extract maximal cliques of size >= k, and merge cliques sharing >= (k-1) nodes.
    This runs in milliseconds even on dense graphs.
    """
    # 1. Collect maximal cliques of size >= k
    cliques: List[Set[str]] = []
    try:
        for c in nx.find_cliques(G):
            if len(c) >= k:
                cliques.append(set(c))
                if len(cliques) >= max_cliques:
                    break
    except Exception:
        cliques = []

    # If no cliques of size k found, try k=3
    if not cliques and k > 3:
        for c in nx.find_cliques(G):
            if len(c) >= 3:
                cliques.append(set(c))
                if len(cliques) >= max_cliques:
                    break
        effective_k = 3
    else:
        effective_k = k

    if not cliques:
        return []

    # 2. Build clique adjacency: connect two cliques if they overlap by >= k-1 nodes
    clique_graph = nx.Graph()
    for i in range(len(cliques)):
        clique_graph.add_node(i)

    overlap_threshold = max(2, effective_k - 1)
    # Sort cliques by size descending to prioritize large cores
    cliques.sort(key=lambda c: len(c), reverse=True)

    # Fast overlap checking
    for i in range(min(len(cliques), 200)):
        ci = cliques[i]
        for j in range(i + 1, min(len(cliques), 200)):
            cj = cliques[j]
            if len(ci.intersection(cj)) >= overlap_threshold:
                clique_graph.add_edge(i, j)

    # 3. Connected components of clique graph form percolation communities
    seed_results: List[Dict[str, Any]] = []
    for comp in nx.connected_components(clique_graph):
        members: Set[str] = set()
        constituent_cliques = []
        for clique_idx in comp:
            c_nodes = cliques[clique_idx]
            members.update(c_nodes)
            if len(constituent_cliques) < 10:
                constituent_cliques.append(list(c_nodes)[:effective_k])

        if len(members) >= effective_k:
            seed_results.append({
                "members": members,
                "cliques": constituent_cliques,
                "k": effective_k,
                "size": len(members),
                "type": f"k-clique (k={effective_k})"
            })

    seed_results.sort(key=lambda x: x["size"], reverse=True)
    return seed_results

**`backend/app/seeding/weak_clique.py`**: Weak-clique seeds (pair + common neighbours, Salton index)

In [ ]:
%%writefile backend/app/seeding/weak_clique.py
import math
from typing import List, Set, Dict, Any, Tuple
import networkx as nx


def salton_similarity(G: nx.Graph, u: str, v: str) -> float:
    """Compute Salton cosine similarity between nodes u and v."""
    nu = set(G.neighbors(u))
    nv = set(G.neighbors(v))
    common = len(nu.intersection(nv))
    denom = math.sqrt(len(nu) * len(nv))
    if denom == 0:
        return 0.0
    return common / denom


def node_priority(G: nx.Graph) -> List[Tuple[str, float]]:
    """
    Compute density-based priority for nodes:
    priority(u) = degree(u) * (clustering_coeff(u) + 0.1)
    """
    clustering = nx.clustering(G)
    degrees = dict(G.degree())
    scores = []
    for node, deg in degrees.items():
        score = deg * (clustering.get(node, 0.0) + 0.1)
        scores.append((node, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


def weak_clique_seeds(G: nx.Graph, min_size: int = 3) -> List[Dict[str, Any]]:
    """
    Extract weak-clique seeds (Paper #26 / WOCD).
    For a linked pair u, v, weak clique is {u, v} U (N(u) cap N(v)).
    """
    priorities = node_priority(G)
    covered: Set[str] = set()
    seeds: List[Dict[str, Any]] = []

    for u, _ in priorities:
        if u in covered:
            continue
        neighbors = list(G.neighbors(u))
        if not neighbors:
            continue

        # Find best neighbor v with highest Salton similarity
        best_v = None
        best_sim = -1.0
        for v in neighbors:
            sim = salton_similarity(G, u, v)
            if sim > best_sim:
                best_sim = sim
                best_v = v

        if best_v is None:
            continue

        nu = set(G.neighbors(u))
        nv = set(G.neighbors(v))
        common = nu.intersection(nv)
        weak_clique = {u, best_v}.union(common)

        if len(weak_clique) >= min_size:
            seeds.append({
                "members": weak_clique,
                "cliques": [[u, best_v] + list(common)[:3]],
                "size": len(weak_clique),
                "type": "weak_clique"
            })
            covered.update(weak_clique)

    seeds.sort(key=lambda x: x["size"], reverse=True)
    return seeds

**`backend/app/seeding/merge.py`**: Jaccard merge of seeds and seed-matrix construction

In [ ]:
%%writefile backend/app/seeding/merge.py
from typing import List, Dict, Any, Tuple
import numpy as np


def jaccard_similarity(s1: set, s2: set) -> float:
    if not s1 and not s2:
        return 1.0
    intersection = len(s1.intersection(s2))
    union = len(s1.union(s2))
    return intersection / union if union > 0 else 0.0


def merge_seeds(seeds: List[Dict[str, Any]], jaccard_threshold: float = 0.6) -> List[Dict[str, Any]]:
    """
    Merge seeds with Jaccard overlap >= jaccard_threshold.
    """
    if not seeds:
        return []

    merged = [dict(s) for s in seeds]
    changed = True

    while changed:
        changed = False
        new_merged = []
        skip_indices = set()

        for i in range(len(merged)):
            if i in skip_indices:
                continue

            current = merged[i]
            cur_members = set(current["members"])
            cur_cliques = list(current.get("cliques", []))

            for j in range(i + 1, len(merged)):
                if j in skip_indices:
                    continue
                other = merged[j]
                other_members = set(other["members"])

                sim = jaccard_similarity(cur_members, other_members)
                if sim >= jaccard_threshold:
                    # Merge j into current
                    cur_members.update(other_members)
                    cur_cliques.extend(other.get("cliques", []))
                    skip_indices.add(j)
                    changed = True

            current["members"] = cur_members
            current["cliques"] = cur_cliques[:10]  # cap stored cliques
            current["size"] = len(cur_members)
            new_merged.append(current)

        merged = new_merged

    merged.sort(key=lambda x: x["size"], reverse=True)
    return merged


def build_seed_matrix(
    seeds: List[Dict[str, Any]],
    node_to_idx: Dict[str, int],
    K: int = 10
) -> Tuple[np.ndarray, List[Dict[str, Any]]]:
    """
    Keep the top K largest seeds and build binary seed matrix S (N x K).
    Returns (S, top_seeds).
    """
    N = len(node_to_idx)
    top_seeds = seeds[:K]
    actual_K = max(1, len(top_seeds))
    S = np.zeros((N, actual_K), dtype=np.float32)

    for c_idx, s in enumerate(top_seeds):
        for node in s["members"]:
            if node in node_to_idx:
                S[node_to_idx[node], c_idx] = 1.0

    return S, top_seeds

**`backend/app/model/__init__.py`**: Model package

In [ ]:
%%writefile backend/app/model/__init__.py
# GNN model package
from .encoder import MembershipGNN
from .losses import bernoulli_poisson_loss, soft_modularity_loss, seed_consistency_loss
from .train import train_model, assign_communities

__all__ = [
    "MembershipGNN",
    "bernoulli_poisson_loss",
    "soft_modularity_loss",
    "seed_consistency_loss",
    "train_model",
    "assign_communities"
]

**`backend/app/model/encoder.py`**: GraphSAGE / GAT membership encoder

In [ ]:
%%writefile backend/app/model/encoder.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class GraphSAGELayer(nn.Module):
    """
    GraphSAGE mean aggregation layer.
    h_v = ReLU(W_self * h_v + W_neigh * Mean_{u in N(v)} h_u)
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.linear_self = nn.Linear(in_features, out_features, bias=bias)
        self.linear_neigh = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x: torch.Tensor, adj_norm: torch.Tensor) -> torch.Tensor:
        # adj_norm is row-normalized adjacency matrix (D^-1 A or similar)
        neigh_x = torch.matmul(adj_norm, x)
        out = self.linear_self(x) + self.linear_neigh(neigh_x)
        return out


class GATLayer(nn.Module):
    """
    Graph Attention Layer with LeakyReLU and softmax attention weights.
    """
    def __init__(self, in_features: int, out_features: int, alpha: float = 0.2):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Parameter(torch.zeros(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
        self.leakyrelu = nn.LeakyReLU(alpha)

    def forward(self, x: torch.Tensor, adj_dense: torch.Tensor) -> torch.Tensor:
        # x: (N, in_features)
        h = self.linear(x)  # (N, out_features)
        N = h.size(0)

        # Self-attention mechanism
        # Concatenate [h_i || h_j] for all pairs
        h_i = h.repeat_interleave(N, dim=0)
        h_j = h.repeat(N, 1)
        a_input = torch.cat([h_i, h_j], dim=1).view(N, N, -1)
        e = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))

        # Mask non-edges
        mask = (adj_dense == 0)
        e = e.masked_fill(mask, -1e9)
        attention = F.softmax(e, dim=1)
        return torch.matmul(attention, h)


class MembershipGNN(nn.Module):
    """
    2-layer GNN encoder mapping [X | S] -> soft community membership matrix F (N x K, >= 0).
    Supported encoder_type: 'sage' (default) or 'gat'.
    """
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        encoder_type: str = "sage",
        dropout: float = 0.2
    ):
        super().__init__()
        self.encoder_type = encoder_type.lower()
        self.dropout = dropout

        if self.encoder_type == "gat":
            self.layer1 = GATLayer(in_dim, hidden_dim)
            self.layer2 = GATLayer(hidden_dim, out_dim)
        else:  # default sage
            self.layer1 = GraphSAGELayer(in_dim, hidden_dim)
            self.layer2 = GraphSAGELayer(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor, adj_norm: torch.Tensor, adj_dense: torch.Tensor) -> torch.Tensor:
        adj_to_use = adj_dense if self.encoder_type == "gat" else adj_norm

        h = self.layer1(x, adj_to_use)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.layer2(h, adj_to_use)
        # Non-negative membership matrix F via ReLU (as in NOCD)
        F_mat = F.relu(h)
        return F_mat

**`backend/app/model/losses.py`**: Bernoulli-Poisson, soft-modularity and seed-consistency losses

In [ ]:
%%writefile backend/app/model/losses.py
import torch
import torch.nn.functional as F


def bernoulli_poisson_loss(
    F_mat: torch.Tensor,
    adj_dense: torch.Tensor,
    neg_ratio: float = 3.0,
    eps: float = 1e-6
) -> torch.Tensor:
    """
    Bernoulli-Poisson link probability loss (standard for overlapping community detection).
    P(edge u,v) = 1 - exp(- F_u . F_v)

    Positive loss on edges: - log(1 - exp(- F_u . F_v))
    Negative loss on non-edges: F_u . F_v
    """
    N = F_mat.size(0)

    # Compute dot products F * F^T
    affinity = torch.matmul(F_mat, F_mat.t())

    # Diagonal elements should not be considered self-edges
    mask_diag = torch.eye(N, dtype=torch.bool, device=F_mat.device)

    # Positive edges mask (upper triangle without diagonal)
    pos_mask = (adj_dense > 0) & (~mask_diag)
    # Negative edges mask
    neg_mask = (adj_dense == 0) & (~mask_diag)

    pos_affinity = affinity[pos_mask]
    if len(pos_affinity) == 0:
        return torch.tensor(0.0, device=F_mat.device, requires_grad=True)

    # Stable -log(1 - exp(-x)):
    # 1 - exp(-x) with clamp
    pos_affinity = torch.clamp(pos_affinity, min=1e-5, max=20.0)
    pos_loss = -torch.log(torch.clamp(1.0 - torch.exp(-pos_affinity), min=eps))
    pos_loss_val = torch.mean(pos_loss)

    # Subsample negative edges to prevent non-edge dominance
    num_neg = int(min(len(pos_affinity) * neg_ratio, neg_mask.sum().item()))
    if num_neg > 0:
        neg_indices = torch.nonzero(neg_mask, as_tuple=False)
        perm = torch.randperm(neg_indices.size(0))[:num_neg]
        selected_neg = neg_indices[perm]
        neg_affinity = affinity[selected_neg[:, 0], selected_neg[:, 1]]
        neg_loss_val = torch.mean(neg_affinity)
    else:
        neg_loss_val = torch.tensor(0.0, device=F_mat.device)

    return pos_loss_val + neg_loss_val


def soft_modularity_loss(
    F_mat: torch.Tensor,
    adj_dense: torch.Tensor
) -> torch.Tensor:
    """
    Differentiable modularity loss:
    Maximizes Tr(S^T B S) / (2m), so loss is -Tr(S^T B S) / (2m).
    """
    degrees = torch.sum(adj_dense, dim=1, keepdim=True)
    m2 = torch.sum(degrees)
    if m2 <= 0:
        return torch.tensor(0.0, device=F_mat.device, requires_grad=True)

    # B = A - (d d^T) / 2m
    B = adj_dense - torch.matmul(degrees, degrees.t()) / m2

    # Softmax over communities for partition representation
    S = F.softmax(F_mat, dim=1)

    # Tr(S^T B S) = sum((S^T B) * S^T)
    # or (S^T @ B @ S).trace()
    SB = torch.matmul(S.t(), B)
    Q = torch.trace(torch.matmul(SB, S)) / m2
    return -Q


def seed_consistency_loss(
    F_mat: torch.Tensor,
    S_mat: torch.Tensor
) -> torch.Tensor:
    """
    Binary Cross-Entropy between squashed F and seed matrix S,
    computed only on nodes that belong to at least one seed.
    """
    if S_mat is None:
        return torch.tensor(0.0, device=F_mat.device, requires_grad=True)

    # Identify nodes in at least one seed
    seeded_mask = torch.sum(S_mat, dim=1) > 0
    if seeded_mask.sum() == 0:
        return torch.tensor(0.0, device=F_mat.device, requires_grad=True)

    # Squash F with sigmoid
    pred_prob = torch.sigmoid(F_mat[seeded_mask])
    target = S_mat[seeded_mask]

    return F.binary_cross_entropy(pred_prob, target)

**`backend/app/model/train.py`**: Training loop and thresholded community assignment

In [ ]:
%%writefile backend/app/model/train.py
import time
from typing import Dict, List, Any, Tuple, Optional
from collections import Counter
import networkx as nx
import numpy as np
import torch
import torch.optim as optim

from .encoder import MembershipGNN
from .losses import bernoulli_poisson_loss, soft_modularity_loss, seed_consistency_loss


def train_model(
    X_t: torch.Tensor,
    S_t: Optional[torch.Tensor],
    adj_norm: torch.Tensor,
    adj_dense: torch.Tensor,
    config: Optional[Dict[str, Any]] = None
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Train the MembershipGNN encoder.

    Returns:
        F_np (np.ndarray): Shape (N, K), learned non-negative membership scores.
        metadata (Dict[str, Any]): Training history and timing.
    """
    cfg = {
        "encoder_type": "sage",
        "hidden_dim": 64,
        "lr": 0.01,
        "weight_decay": 1e-4,
        "epochs": 150,
        "structure_loss": "bp",  # 'bp' or 'modularity'
        "lambda_seed": 1.0,
        "dropout": 0.1,
    }
    if config:
        cfg.update(config)

    N = X_t.size(0)
    K = S_t.size(1) if S_t is not None else 10

    # Node input is [X | S] so the GNN starts from clique structure
    if S_t is not None:
        node_input = torch.cat([X_t, S_t], dim=1)
    else:
        node_input = X_t

    in_dim = node_input.size(1)
    model = MembershipGNN(
        in_dim=in_dim,
        hidden_dim=cfg["hidden_dim"],
        out_dim=K,
        encoder_type=cfg["encoder_type"],
        dropout=cfg["dropout"]
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    start_time = time.time()
    losses = []

    model.train()
    for epoch in range(cfg["epochs"]):
        optimizer.zero_grad()
        F_mat = model(node_input, adj_norm, adj_dense)

        # Structure loss
        if cfg["structure_loss"] == "modularity":
            l_struct = soft_modularity_loss(F_mat, adj_dense)
        else:
            l_struct = bernoulli_poisson_loss(F_mat, adj_dense)

        # Seed consistency loss
        if S_t is not None and cfg["lambda_seed"] > 0:
            l_seed = seed_consistency_loss(F_mat, S_t)
        else:
            l_seed = torch.tensor(0.0)

        total_loss = l_struct + cfg["lambda_seed"] * l_seed

        if torch.isnan(total_loss):
            break

        total_loss.backward()
        optimizer.step()
        losses.append(total_loss.item())

    total_time = time.time() - start_time

    model.eval()
    with torch.no_grad():
        F_final = model(node_input, adj_norm, adj_dense)
        F_np = F_final.cpu().numpy()

    metadata = {
        "losses": losses,
        "final_loss": losses[-1] if losses else 0.0,
        "train_time_sec": round(total_time, 4),
        "config": cfg,
        "K": K
    }
    return F_np, metadata


def assign_communities(
    F_np: np.ndarray,
    G: nx.Graph,
    node_list: List[str],
    tau: float = 0.5,
    fill_unassigned: bool = True
) -> Dict[str, List[str]]:
    """
    Threshold membership matrix F to assign nodes to communities.
    Unassigned nodes take the most common circle among their neighbors (Paper #20).

    Returns:
        communities: Dict mapping 'circle_c' -> list of node IDs.
    """
    N, K = F_np.shape
    node_to_idx = {nid: i for i, nid in enumerate(node_list)}

    # Binary assignments
    assignments: List[List[int]] = [[] for _ in range(N)]
    for i in range(N):
        for c in range(K):
            if F_np[i, c] >= tau:
                assignments[i].append(c)

    # Fill unassigned nodes using neighbor votes
    if fill_unassigned:
        for i, nid in enumerate(node_list):
            if not assignments[i] and nid in G:
                neighbor_circles = []
                for neighbor in G.neighbors(nid):
                    if neighbor in node_to_idx:
                        n_idx = node_to_idx[neighbor]
                        neighbor_circles.extend(assignments[n_idx])
                if neighbor_circles:
                    most_common_c = Counter(neighbor_circles).most_common(1)[0][0]
                    assignments[i].append(most_common_c)
                else:
                    # Fallback to node's highest scoring column in F
                    best_c = int(np.argmax(F_np[i]))
                    assignments[i].append(best_c)

    # Invert into dict of communities
    communities: Dict[str, List[str]] = {}
    for c in range(K):
        c_name = f"circle_{c}"
        members = [node_list[i] for i in range(N) if c in assignments[i]]
        if members:  # only include non-empty circles
            communities[c_name] = members

    return communities

**`backend/app/baselines/__init__.py`**: Baselines package

In [ ]:
%%writefile backend/app/baselines/__init__.py
# Baselines package
from .run import run_cpm, run_leiden_louvain, run_bigclam, run_gnn_no_seeds

__all__ = ["run_cpm", "run_leiden_louvain", "run_bigclam", "run_gnn_no_seeds"]

**`backend/app/baselines/run.py`**: Baselines: CPM, Louvain, BigCLAM, GNN without seeds

In [ ]:
%%writefile backend/app/baselines/run.py
import time
from typing import Dict, List, Any, Tuple
import networkx as nx
import numpy as np
import torch

from ..seeding.cpm import k_clique_seeds
from ..model.train import train_model, assign_communities


def run_cpm(G: nx.Graph, k: int = 4) -> Tuple[Dict[str, List[str]], float]:
    """
    Run baseline Clique Percolation Method (CPM).
    Returns (communities_dict, runtime_sec).
    """
    t0 = time.time()
    raw_seeds = k_clique_seeds(G, k=k)
    communities = {}
    for i, s in enumerate(raw_seeds):
        communities[f"circle_{i}"] = sorted(list(s["members"]))
    elapsed = time.time() - t0
    return communities, round(elapsed, 4)


def run_leiden_louvain(G: nx.Graph) -> Tuple[Dict[str, List[str]], float]:
    """
    Run baseline disjoint community detection (Louvain/Leiden modularity).
    Returns (communities_dict, runtime_sec).
    """
    t0 = time.time()
    try:
        raw_comms = nx.community.louvain_communities(G, seed=42)
    except Exception:
        raw_comms = list(nx.connected_components(G))

    communities = {}
    for i, c in enumerate(raw_comms):
        communities[f"circle_{i}"] = sorted(list(c))
    elapsed = time.time() - t0
    return communities, round(elapsed, 4)


def run_bigclam(
    G: nx.Graph,
    node_list: List[str],
    K: int = 10,
    iterations: int = 50,
    lr: float = 0.05
) -> Tuple[Dict[str, List[str]], float]:
    """
    Run baseline BigCLAM (Overlapping Community Detection via Non-negative Factorization).
    Models P(u,v) = 1 - exp(- F_u . F_v).
    """
    t0 = time.time()
    N = len(node_list)
    node_to_idx = {nid: i for i, nid in enumerate(node_list)}

    # Initialize F randomly >= 0
    np.random.seed(42)
    F_mat = np.random.uniform(0.01, 0.1, size=(N, K)).astype(np.float32)

    # Pre-extract adjacency list for fast updates
    adj_set = {i: set() for i in range(N)}
    for u, v in G.edges():
        if u in node_to_idx and v in node_to_idx:
            ui, vi = node_to_idx[u], node_to_idx[v]
            adj_set[ui].add(vi)
            adj_set[vi].add(ui)

    # Simplified gradient descent update for BigCLAM
    # grad_u = sum_{v in N(u)} (F_v * exp(-F_u . F_v) / (1 - exp(-F_u . F_v))) - sum_{v not in N(u)} F_v
    for _ in range(iterations):
        sum_F = np.sum(F_mat, axis=0)  # (K,)
        for u in range(N):
            neighs = list(adj_set[u])
            if not neighs:
                continue

            F_u = F_mat[u]
            F_neigh = F_mat[neighs]  # (|N|, K)
            dots = np.dot(F_neigh, F_u)
            dots = np.clip(dots, 1e-4, 15.0)

            # Positive gradient from neighbors
            factors = np.exp(-dots) / (1.0 - np.exp(-dots) + 1e-6)
            factors = np.clip(factors, 0.0, 1e4)
            pos_grad = np.sum(F_neigh * factors[:, None], axis=0)

            # Negative gradient: sum_{v not in N(u)} F_v = sum_F - F_u - sum_{v in N(u)} F_v
            neg_grad = sum_F - F_u - np.sum(F_neigh, axis=0)

            grad = pos_grad - neg_grad
            # Gradient ascent step with safe clipping
            F_mat[u] = np.clip(F_u + lr * (grad / (len(neighs) + 1.0)), 0.0, 5.0)

    # Threshold membership
    tau = np.sqrt(-np.log(1.0 - 0.5)) if False else 0.2
    communities = {}
    for c in range(K):
        members = [node_list[i] for i in range(N) if F_mat[i, c] >= tau]
        if members:
            communities[f"circle_{c}"] = members

    # If any node unassigned, give to highest column
    assigned_nodes = set().union(*communities.values()) if communities else set()
    unassigned = [node_list[i] for i in range(N) if node_list[i] not in assigned_nodes]
    for un in unassigned:
        best_c = int(np.argmax(F_mat[node_to_idx[un]]))
        c_name = f"circle_{best_c}"
        if c_name not in communities:
            communities[c_name] = []
        communities[c_name].append(un)

    elapsed = time.time() - t0
    return communities, round(elapsed, 4)


def run_gnn_no_seeds(
    X_t: torch.Tensor,
    adj_norm: torch.Tensor,
    adj_dense: torch.Tensor,
    G: nx.Graph,
    node_list: List[str],
    K: int = 10,
    tau: float = 0.5
) -> Tuple[Dict[str, List[str]], float]:
    """
    Run ablation baseline: GNN without clique seeds (pure NOCD-style).
    """
    t0 = time.time()
    cfg = {
        "encoder_type": "sage",
        "hidden_dim": 64,
        "lr": 0.01,
        "epochs": 150,
        "structure_loss": "bp",
        "lambda_seed": 0.0,
    }
    # Pass dummy zero seed or None
    S_dummy = torch.zeros((X_t.size(0), K), dtype=torch.float32)
    F_np, _ = train_model(X_t, S_dummy, adj_norm, adj_dense, cfg)
    comms = assign_communities(F_np, G, node_list, tau=tau)
    elapsed = time.time() - t0
    return comms, round(elapsed, 4)

**`backend/app/evaluation/__init__.py`**: Evaluation package

In [ ]:
%%writefile backend/app/evaluation/__init__.py
# Evaluation metrics package
from .metrics import (
    overlapping_nmi,
    average_f1,
    omega_index,
    bridge_node_scores,
    community_quality,
    evaluate_all
)

__all__ = [
    "overlapping_nmi",
    "average_f1",
    "omega_index",
    "bridge_node_scores",
    "community_quality",
    "evaluate_all"
]

**`backend/app/evaluation/metrics.py`**: ONMI, average F1, Omega index, bridge F1, community quality

In [ ]:
%%writefile backend/app/evaluation/metrics.py
import math
from typing import Dict, List, Set, Tuple, Any
from itertools import combinations
import numpy as np
import networkx as nx


def _f1_pair(set_a: Set[str], set_b: Set[str]) -> float:
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    common = len(set_a.intersection(set_b))
    precision = common / len(set_a)
    recall = common / len(set_b)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def average_f1(pred_dict: Dict[str, List[str]], truth_dict: Dict[str, List[str]]) -> float:
    """
    Compute average F1 score between predicted and ground-truth communities
    with bidirectional best-match pairing (standard in SNAP ego-Facebook benchmarks).
    """
    pred_sets = [set(v) for v in pred_dict.values() if len(v) > 0]
    truth_sets = [set(v) for v in truth_dict.values() if len(v) > 0]

    if not pred_sets or not truth_sets:
        return 0.0

    # Match each truth community to best pred community
    f1_truth_to_pred = []
    for t in truth_sets:
        best_f1 = max((_f1_pair(t, p) for p in pred_sets), default=0.0)
        f1_truth_to_pred.append(best_f1)

    # Match each pred community to best truth community
    f1_pred_to_truth = []
    for p in pred_sets:
        best_f1 = max((_f1_pair(p, t) for t in truth_sets), default=0.0)
        f1_pred_to_truth.append(best_f1)

    score = 0.5 * (np.mean(f1_truth_to_pred) + np.mean(f1_pred_to_truth))
    return float(score)


def overlapping_nmi(
    pred_dict: Dict[str, List[str]],
    truth_dict: Dict[str, List[str]],
    all_nodes: List[str]
) -> float:
    """
    Overlapping Normalized Mutual Information (ONMI) based on Lancichinetti et al.
    For each community pair, computes Shannon mutual information over binary membership.
    Normalized by average entropy.
    """
    pred_sets = [set(v) for v in pred_dict.values() if len(v) > 0]
    truth_sets = [set(v) for v in truth_dict.values() if len(v) > 0]

    if not pred_sets or not truth_sets or not all_nodes:
        return 0.0

    N = len(all_nodes)

    def binary_entropy(s: Set[str]) -> float:
        p1 = len(s) / N
        p0 = 1.0 - p1
        h = 0.0
        if p1 > 0:
            h -= p1 * math.log2(p1)
        if p0 > 0:
            h -= p0 * math.log2(p0)
        return h

    def mi_pair(s1: Set[str], s2: Set[str]) -> float:
        # 2x2 contingency table
        n11 = len(s1.intersection(s2))
        n10 = len(s1) - n11
        n01 = len(s2) - n11
        n00 = N - (n11 + n10 + n01)

        counts = [n11, n10, n01, n00]
        p_row1 = len(s1) / N
        p_row0 = 1.0 - p_row1
        p_col1 = len(s2) / N
        p_col0 = 1.0 - p_col1

        marginals = [
            p_row1 * p_col1,
            p_row1 * p_col0,
            p_row0 * p_col1,
            p_row0 * p_col0
        ]

        mi = 0.0
        for c, m in zip(counts, marginals):
            if c > 0 and m > 0:
                p_joint = c / N
                mi += p_joint * math.log2(p_joint / m)
        return max(0.0, mi)

    h_truth = sum(binary_entropy(t) for t in truth_sets)
    h_pred = sum(binary_entropy(p) for p in pred_sets)

    if h_truth == 0.0 or h_pred == 0.0:
        return 0.0

    # Best-match mutual information sum
    mi_truth = sum(max(mi_pair(t, p) for p in pred_sets) for t in truth_sets)
    mi_pred = sum(max(mi_pair(p, t) for t in truth_sets) for p in pred_sets)

    total_mi = 0.5 * (mi_truth + mi_pred)
    norm = 0.5 * (h_truth + h_pred)

    onmi_val = total_mi / norm if norm > 0 else 0.0
    return float(np.clip(onmi_val, 0.0, 1.0))


def omega_index(
    pred_dict: Dict[str, List[str]],
    truth_dict: Dict[str, List[str]],
    all_nodes: List[str],
    sample_limit: int = 1000
) -> float:
    """
    Omega Index measures agreement between two overlapping clusterings
    based on the number of shared communities for each pair of nodes.
    Omega = (observed_agreement - expected_agreement) / (1 - expected_agreement)
    """
    pred_sets = [set(v) for v in pred_dict.values()]
    truth_sets = [set(v) for v in truth_dict.values()]

    nodes = list(all_nodes)
    if len(nodes) > sample_limit:
        # Sample to keep computation fast on very large networks
        np.random.seed(42)
        nodes = list(np.random.choice(nodes, size=sample_limit, replace=False))

    N = len(nodes)
    if N < 2:
        return 1.0

    pairs_total = N * (N - 1) // 2

    # Map each node to its community indices
    pred_map = {n: set() for n in nodes}
    for c_idx, p in enumerate(pred_sets):
        for n in p:
            if n in pred_map:
                pred_map[n].add(c_idx)

    truth_map = {n: set() for n in nodes}
    for c_idx, t in enumerate(truth_sets):
        for n in t:
            if n in truth_map:
                truth_map[n].add(c_idx)

    matches = 0
    t_counts = {}
    p_counts = {}

    for i in range(N):
        u = nodes[i]
        u_p = pred_map[u]
        u_t = truth_map[u]
        for j in range(i + 1, N):
            v = nodes[j]
            k_pred = len(u_p.intersection(pred_map[v]))
            k_truth = len(u_t.intersection(truth_map[v]))

            p_counts[k_pred] = p_counts.get(k_pred, 0) + 1
            t_counts[k_truth] = t_counts.get(k_truth, 0) + 1

            if k_pred == k_truth:
                matches += 1

    obs = matches / pairs_total

    # Expected agreement by chance
    exp = 0.0
    all_k = set(p_counts.keys()).union(t_counts.keys())
    for k in all_k:
        cp = p_counts.get(k, 0) / pairs_total
        ct = t_counts.get(k, 0) / pairs_total
        exp += cp * ct

    if exp >= 1.0:
        return 1.0
    omega = (obs - exp) / (1.0 - exp)
    return float(np.clip(omega, -1.0, 1.0))


def bridge_node_scores(
    pred_dict: Dict[str, List[str]],
    truth_dict: Dict[str, List[str]]
) -> Dict[str, float]:
    """
    Evaluate detection of bridge (overlapping) nodes.
    A node is a bridge if it belongs to >= 2 communities.
    """
    pred_node_counts = {}
    for members in pred_dict.values():
        for n in members:
            pred_node_counts[n] = pred_node_counts.get(n, 0) + 1

    truth_node_counts = {}
    for members in truth_dict.values():
        for n in members:
            truth_node_counts[n] = truth_node_counts.get(n, 0) + 1

    pred_bridges = {n for n, cnt in pred_node_counts.items() if cnt >= 2}
    truth_bridges = {n for n, cnt in truth_node_counts.items() if cnt >= 2}

    if not truth_bridges and not pred_bridges:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}

    common = len(pred_bridges.intersection(truth_bridges))
    prec = common / len(pred_bridges) if pred_bridges else 0.0
    rec = common / len(truth_bridges) if truth_bridges else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

    return {
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "true_bridges_count": len(truth_bridges),
        "pred_bridges_count": len(pred_bridges)
    }


def community_quality(G: nx.Graph, members: List[str]) -> Dict[str, float]:
    """
    Compute internal density and conductance of a community.
    """
    k = len(members)
    if k <= 1:
        return {"density": 0.0, "conductance": 0.0, "size": k}

    member_set = set(members)
    subG = G.subgraph(member_set)
    in_edges = subG.number_of_edges()
    max_possible = k * (k - 1) / 2
    density = (in_edges / max_possible) if max_possible > 0 else 0.0

    cut_edges = 0
    for u in member_set:
        if u in G:
            for v in G.neighbors(u):
                if v not in member_set:
                    cut_edges += 1

    vol = (2 * in_edges) + cut_edges
    conductance = (cut_edges / vol) if vol > 0 else 0.0

    return {
        "density": round(float(density), 4),
        "conductance": round(float(conductance), 4),
        "size": k
    }


def evaluate_all(
    pred_dict: Dict[str, List[str]],
    truth_dict: Dict[str, List[str]],
    all_nodes: List[str]
) -> Dict[str, Any]:
    """
    Run full metric suite: ONMI, Average F1, Omega Index, Bridge scores.
    """
    f1 = average_f1(pred_dict, truth_dict)
    nmi = overlapping_nmi(pred_dict, truth_dict, all_nodes)
    omega = omega_index(pred_dict, truth_dict, all_nodes)
    bridge_metrics = bridge_node_scores(pred_dict, truth_dict)

    return {
        "avg_f1": round(f1, 4),
        "onmi": round(nmi, 4),
        "omega": round(omega, 4),
        "bridge_precision": bridge_metrics["precision"],
        "bridge_recall": bridge_metrics["recall"],
        "bridge_f1": bridge_metrics["f1"]
    }

**`backend/app/analysis/__init__.py`**: Analysis package

In [ ]:
%%writefile backend/app/analysis/__init__.py
# Analysis package
from .bridges import find_bridge_friends
from .recommend import suggest_friends, evaluate_suggestions
from .explain import explain_circle

__all__ = ["find_bridge_friends", "suggest_friends", "evaluate_suggestions", "explain_circle"]

**`backend/app/analysis/bridges.py`**: Bridge-friend detection

In [ ]:
%%writefile backend/app/analysis/bridges.py
from typing import List, Dict, Any
import numpy as np


def find_bridge_friends(
    F_np: np.ndarray,
    node_list: List[str],
    tau: float = 0.5,
    top_n: int = 50
) -> List[Dict[str, Any]]:
    """
    Identify and rank bridge friends who belong to two or more circles.
    Ranked by number of circles, then by aggregate membership strength.
    """
    N, K = F_np.shape
    bridges = []

    for i in range(N):
        node_id = node_list[i]
        scores = F_np[i]

        active_circles = []
        for c in range(K):
            if scores[c] >= tau:
                active_circles.append({
                    "circle_id": c,
                    "circle_name": f"circle_{c}",
                    "score": round(float(scores[c]), 4)
                })

        # Multi-circle members
        if len(active_circles) >= 2:
            total_strength = float(np.sum([c["score"] for c in active_circles]))
            bridges.append({
                "node_id": node_id,
                "node_index": i,
                "circle_count": len(active_circles),
                "total_strength": round(total_strength, 4),
                "circles": active_circles
            })

    # Sort descending by circle_count, then total_strength
    bridges.sort(key=lambda x: (x["circle_count"], x["total_strength"]), reverse=True)
    return bridges[:top_n]

**`backend/app/analysis/recommend.py`**: Friend suggestions from circle affinity

In [ ]:
%%writefile backend/app/analysis/recommend.py
from typing import List, Dict, Any, Optional
import networkx as nx
import numpy as np


def suggest_friends(
    F_np: np.ndarray,
    G: nx.Graph,
    user_id: str,
    node_list: List[str],
    node_to_idx: Dict[str, int],
    top_n: int = 10
) -> List[Dict[str, Any]]:
    """
    Suggest new friends to user_id based on shared-circle affinity (F_u . F_v).
    Excludes existing friends and the user themself.
    """
    if user_id not in node_to_idx:
        return []

    u_idx = node_to_idx[user_id]
    u_vec = F_np[u_idx]

    # Current neighbors
    existing_friends = set(G.neighbors(user_id)) if user_id in G else set()
    existing_friends.add(user_id)

    suggestions = []
    N, K = F_np.shape

    for v_nid in node_list:
        if v_nid in existing_friends:
            continue
        v_idx = node_to_idx[v_nid]
        v_vec = F_np[v_idx]

        # Dot product affinity
        dot_score = float(np.dot(u_vec, v_vec))
        if dot_score <= 1e-4:
            continue

        # Detail shared circles
        shared_circles = []
        for c in range(K):
            contrib = float(u_vec[c] * v_vec[c])
            if contrib > 0.01:
                shared_circles.append({
                    "circle_id": c,
                    "circle_name": f"circle_{c}",
                    "affinity_contribution": round(contrib, 4),
                    "user_score": round(float(u_vec[c]), 4),
                    "friend_score": round(float(v_vec[c]), 4)
                })

        shared_circles.sort(key=lambda x: x["affinity_contribution"], reverse=True)

        suggestions.append({
            "target_node": v_nid,
            "affinity_score": round(dot_score, 4),
            "shared_circles": shared_circles,
            "common_neighbors_count": len(existing_friends.intersection(set(G.neighbors(v_nid)))) if v_nid in G else 0
        })

    suggestions.sort(key=lambda x: x["affinity_score"], reverse=True)
    return suggestions[:top_n]


def evaluate_suggestions(
    G: nx.Graph,
    F_np: np.ndarray,
    node_list: List[str],
    node_to_idx: Dict[str, int],
    hide_ratio: float = 0.1,
    top_k: int = 10
) -> Dict[str, float]:
    """
    Evaluate recommendation quality by hiding 10% of edges and measuring
    Hit-Rate@K against a standard Common-Neighbors baseline.
    """
    edges = list(G.edges())
    num_hide = int(len(edges) * hide_ratio)
    if num_hide == 0:
        return {"hybrid_hit_rate": 0.0, "cn_hit_rate": 0.0}

    np.random.seed(42)
    hide_idx = np.random.choice(len(edges), size=num_hide, replace=False)
    hidden_edges = set(edges[i] for i in hide_idx)

    # Subgraph without hidden edges
    G_train = G.copy()
    G_train.remove_edges_from(hidden_edges)

    hits_hybrid = 0
    hits_cn = 0
    test_nodes = set()
    for u, v in hidden_edges:
        test_nodes.add(u)

    eval_count = 0
    for u in list(test_nodes)[:30]:  # evaluate on sample for speed
        if u not in node_to_idx:
            continue
        eval_count += 1
        true_hidden_neighbors = {v for v in G.neighbors(u) if (u, v) in hidden_edges or (v, u) in hidden_edges}

        # Hybrid predictions
        recs_hybrid = suggest_friends(F_np, G_train, u, node_list, node_to_idx, top_n=top_k)
        pred_nodes_hybrid = {r["target_node"] for r in recs_hybrid}
        if true_hidden_neighbors.intersection(pred_nodes_hybrid):
            hits_hybrid += 1

        # Common neighbors baseline
        cn_scores = []
        u_neighs = set(G_train.neighbors(u)) if u in G_train else set()
        for v in node_list:
            if v != u and not G_train.has_edge(u, v):
                v_neighs = set(G_train.neighbors(v)) if v in G_train else set()
                cn_scores.append((v, len(u_neighs.intersection(v_neighs))))
        cn_scores.sort(key=lambda x: x[1], reverse=True)
        top_cn_nodes = {x[0] for x in cn_scores[:top_k]}
        if true_hidden_neighbors.intersection(top_cn_nodes):
            hits_cn += 1

    return {
        "hybrid_hit_rate": round(hits_hybrid / max(1, eval_count), 4),
        "cn_hit_rate": round(hits_cn / max(1, eval_count), 4),
        "evaluated_nodes": eval_count
    }

**`backend/app/analysis/explain.py`**: Circle explanation via seed cliques and features

In [ ]:
%%writefile backend/app/analysis/explain.py
from typing import Dict, List, Any, Optional
import networkx as nx
import numpy as np

from ..evaluation.metrics import community_quality


def explain_circle(
    circle_idx: int,
    top_seeds: List[Dict[str, Any]],
    G: nx.Graph,
    communities: Dict[str, List[str]],
    X: Optional[np.ndarray] = None,
    node_to_idx: Optional[Dict[str, int]] = None,
    feat_names: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Explain a learned circle:
    - Which clique seed it originated from
    - The constituent cliques
    - Density, conductance, and size
    - Most shared/distinctive profile features
    """
    circle_key = f"circle_{circle_idx}"
    members = communities.get(circle_key, [])

    # Retrieve corresponding seed if within range
    seed_info = {}
    if circle_idx < len(top_seeds):
        s = top_seeds[circle_idx]
        seed_info = {
            "seed_id": circle_idx,
            "seed_members": sorted(list(s.get("members", []))),
            "seed_size": s.get("size", len(s.get("members", []))),
            "constituent_cliques": s.get("cliques", []),
            "seed_type": s.get("type", f"k-clique (k={s.get('k', 4)})")
        }

    # Graph quality metrics
    quality = community_quality(G, members)

    # Feature prominence
    top_features = []
    if X is not None and node_to_idx is not None and len(members) > 0:
        member_indices = [node_to_idx[m] for m in members if m in node_to_idx]
        if member_indices:
            sub_X = X[member_indices]
            # Frequency in this circle vs overall
            circle_freq = np.mean(sub_X, axis=0)
            overall_freq = np.mean(X, axis=0)
            # Distinctiveness ratio: circle_freq - overall_freq
            distinct = circle_freq - overall_freq
            best_feat_idx = np.argsort(distinct)[::-1][:5]

            for fid in best_feat_idx:
                if circle_freq[fid] > 0.05:  # present in at least 5% of circle
                    fname = feat_names[fid] if (feat_names and fid < len(feat_names)) else f"Feature #{fid}"
                    top_features.append({
                        "feature_id": int(fid),
                        "name": fname,
                        "circle_frequency": round(float(circle_freq[fid]), 3),
                        "overall_frequency": round(float(overall_freq[fid]), 3),
                        "distinctiveness": round(float(distinct[fid]), 3)
                    })

    return {
        "circle_id": circle_idx,
        "circle_name": circle_key,
        "size": len(members),
        "members": members,
        "quality": quality,
        "seed_origin": seed_info,
        "top_features": top_features,
        "summary": (
            f"Circle {circle_idx} contains {len(members)} friends. "
            f"It originated from a {seed_info.get('seed_type', 'clique')} seed of {seed_info.get('seed_size', 'N/A')} nodes "
            f"with internal density of {quality['density']} and conductance of {quality['conductance']}."
        )
    }


_PRETTY = {
    "education;school": "School",
    "education;year": "Graduation-year",
    "education;type": "Education-level",
    "education;concentration": "Study-field",
    "education;degree": "Degree",
    "work;employer": "Workplace",
    "work;position": "Job-role",
    "work;location": "Work-location",
    "work;start_date": "Work-period",
    "work;end_date": "Work-period",
    "location": "Location",
    "hometown": "Hometown",
    "languages": "Language",
    "last_name": "Family (shared surname)",
    "first_name": "First-name",
    "birthday": "Birthday",
    "gender": "Gender",
    "locale": "Locale",
    "political": "Political-view",
    "religion": "Religion",
    "quotes": "Quotes",
}


_UNINFORMATIVE = {"gender", "locale", "first_name", "birthday", "quotes"}


def _feature_category(name: str) -> str:
    """'166 work;end_date;anonymized feature 163' -> 'work;end_date'."""
    parts = name.strip().split(" ", 1)
    if len(parts) == 2 and parts[0].isdigit():
        name = parts[1]
    tokens = [t for t in name.split(";") if t and t != "id" and not t.startswith("anonymized")]
    return ";".join(tokens)


def label_circles(
    communities: Dict[str, List[str]],
    X: np.ndarray,
    node_to_idx: Dict[str, int],
    feat_names: List[str],
    min_freq: float = 0.25,
    min_lift: float = 0.15,
) -> Dict[str, str]:
    """
    Give each circle a readable name from the profile-feature category that is most
    over-represented among its members (e.g. 'Workplace group').
    Feature values are anonymised in the dataset, so we can name the kind of tie
    (school, workplace, family surname) but not the actual school or employer.
    """
    if X is None or not feat_names:
        return {c: c.replace("_", " ").title() for c in communities}

    categories = [_feature_category(n) for n in feat_names[: X.shape[1]]]
    overall = X.mean(axis=0)
    labels: Dict[str, str] = {}
    used: Dict[str, int] = {}

    for cname, members in communities.items():
        idx = [node_to_idx[m] for m in members if m in node_to_idx]
        base = None
        if len(idx) >= 3:
            freq = X[idx].mean(axis=0)
            lift = freq - overall
            best_score, best_cat = 0.0, None
            for f, cat in enumerate(categories):
                if cat in _UNINFORMATIVE:
                    continue
                if freq[f] >= min_freq and lift[f] >= min_lift and lift[f] > best_score:
                    best_score, best_cat = float(lift[f]), cat
            if best_cat:
                base = f"{_PRETTY.get(best_cat, best_cat.replace(';', ' ').replace('_', ' ').title())} group"
        if base is None:
            base = "Mixed circle"
        used[base] = used.get(base, 0) + 1
        labels[cname] = base if used[base] == 1 else f"{base} #{used[base]}"
    # Number the first occurrence too when a name repeats, so labels are unambiguous
    for base, count in used.items():
        if count > 1:
            first = next(c for c, l in labels.items() if l == base)
            labels[first] = f"{base} #1"
    return labels

**`backend/app/pipeline.py`**: End-to-end pipeline `run_pipeline`

In [ ]:
%%writefile backend/app/pipeline.py
import os
import json
import time
from typing import Dict, Any, Optional
import numpy as np

from .data.loader import load_ego, to_graph_tensors, get_data_dir
from .seeding.cpm import k_clique_seeds
from .seeding.weak_clique import weak_clique_seeds
from .seeding.merge import merge_seeds, build_seed_matrix
from .model.train import train_model, assign_communities
from .baselines.run import run_cpm, run_leiden_louvain, run_bigclam, run_gnn_no_seeds
from .evaluation.metrics import evaluate_all, community_quality
from .analysis.bridges import find_bridge_friends
from .analysis.recommend import suggest_friends, evaluate_suggestions
from .analysis.explain import explain_circle


def get_results_dir() -> str:
    curr = os.path.abspath(os.path.dirname(__file__))
    candidates = [
        os.path.join(curr, "..", "results"),
        os.path.join(curr, "..", "..", "results"),
        os.path.join(os.getcwd(), "backend", "results"),
        os.path.join(os.getcwd(), "results"),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    target = os.path.abspath(candidates[0])
    os.makedirs(target, exist_ok=True)
    return target


def run_pipeline(
    ego_id: int,
    config: Optional[Dict[str, Any]] = None,
    save_results: bool = True
) -> Dict[str, Any]:
    """
    Run full CircleSense pipeline for an ego-network:
    1. Load data
    2. Seeding (CPM or Weak-clique)
    3. GNN training & soft-membership output
    4. Community assignment & bridge detection
    5. Baselines execution
    6. Full metric evaluation
    7. Recommendation & explanation generation
    8. Serialize results
    """
    t_start = time.time()
    cfg = {
        "k": 4,
        "seeding": "weak_clique",  # 'cpm' or 'weak_clique'
        "encoder_type": "sage",  # 'sage' or 'gat'
        "hidden_dim": 64,
        "lr": 0.01,
        "epochs": 300,
        "structure_loss": "bp",  # 'bp' or 'modularity'
        "lambda_seed": 1.0,
        "tau": 0.5,
        "jaccard_merge": 0.6,
        "max_seeds": 20
    }
    if config:
        cfg.update(config)

    # 1. Load data
    G, node_list, node_to_idx, X, ground_truth, feat_names = load_ego(ego_id)
    N = len(node_list)

    # 2. Seeding
    t_seed_start = time.time()
    if cfg["seeding"] == "weak_clique":
        raw_seeds = weak_clique_seeds(G)
    else:
        raw_seeds = k_clique_seeds(G, k=cfg["k"])

    merged_seeds = merge_seeds(raw_seeds, jaccard_threshold=cfg["jaccard_merge"])

    # Determine K: use number of ground-truth circles or max_seeds
    K = max(3, min(len(ground_truth) if ground_truth else 10, len(merged_seeds), cfg["max_seeds"]))
    S_np, top_seeds = build_seed_matrix(merged_seeds, node_to_idx, K=K)
    seed_time = round(time.time() - t_seed_start, 4)

    # 3. Graph tensors
    edge_index, adj_dense, adj_norm, X_t, S_t = to_graph_tensors(G, X, S_np)

    # 4. GNN Training
    F_np, train_meta = train_model(X_t, S_t, adj_norm, adj_dense, cfg)

    # 5. Community Assignment
    hybrid_comms = assign_communities(F_np, G, node_list, tau=cfg["tau"])

    # 6. Baselines
    cpm_comms, cpm_time = run_cpm(G, k=cfg["k"])
    leiden_comms, leiden_time = run_leiden_louvain(G)
    bigclam_comms, bigclam_time = run_bigclam(G, node_list, K=K)
    gnn_noseed_comms, gnn_noseed_time = run_gnn_no_seeds(
        X_t, adj_norm, adj_dense, G, node_list, K=K, tau=cfg["tau"]
    )

    # 7. Metrics evaluation on ground truth
    metrics_hybrid = evaluate_all(hybrid_comms, ground_truth, node_list)
    metrics_cpm = evaluate_all(cpm_comms, ground_truth, node_list)
    metrics_leiden = evaluate_all(leiden_comms, ground_truth, node_list)
    metrics_bigclam = evaluate_all(bigclam_comms, ground_truth, node_list)
    metrics_gnn_noseed = evaluate_all(gnn_noseed_comms, ground_truth, node_list)

    metrics_comparison = {
        "hybrid": {**metrics_hybrid, "runtime": train_meta["train_time_sec"] + seed_time},
        "cpm": {**metrics_cpm, "runtime": cpm_time},
        "leiden": {**metrics_leiden, "runtime": leiden_time},
        "bigclam": {**metrics_bigclam, "runtime": bigclam_time},
        "gnn_no_seeds": {**metrics_gnn_noseed, "runtime": gnn_noseed_time}
    }

    # 8. Analysis: Bridges & Explanations
    bridges = find_bridge_friends(F_np, node_list, tau=cfg["tau"], top_n=30)

    explanations = {}
    for c_idx in range(K):
        c_key = f"circle_{c_idx}"
        if c_key in hybrid_comms:
            explanations[c_key] = explain_circle(
                c_idx, top_seeds, G, hybrid_comms, X, node_to_idx, feat_names
            )

    # Recommendation evaluations
    rec_eval = evaluate_suggestions(G, F_np, node_list, node_to_idx, hide_ratio=0.1)

    # Graph visualization format (Cytoscape.js)
    # Map each node to its circles across methods
    node_circles_map = {n: [] for n in node_list}
    for c_name, members in hybrid_comms.items():
        for m in members:
            node_circles_map[m].append(c_name)

    bridge_set = {b["node_id"] for b in bridges}

    cytoscape_elements = []
    degrees = dict(G.degree())
    for nid in node_list:
        assigned_c = node_circles_map.get(nid, [])
        cytoscape_elements.append({
            "data": {
                "id": str(nid),
                "label": f"Friend {nid}",
                "degree": degrees.get(nid, 0),
                "circles": assigned_c,
                "is_bridge": nid in bridge_set,
                "circle_count": len(assigned_c)
            }
        })

    for u, v in G.edges():
        cytoscape_elements.append({
            "data": {
                "id": f"e_{u}_{v}",
                "source": str(u),
                "target": str(v)
            }
        })

    # Summary payload
    results_payload = {
        "ego_id": ego_id,
        "config": cfg,
        "graph_summary": {
            "num_nodes": N,
            "num_edges": G.number_of_edges(),
            "num_ground_truth_circles": len(ground_truth),
            "num_detected_circles": len(hybrid_comms),
            "num_bridges": len(bridges),
            "features_dim": X.shape[1]
        },
        "metrics": metrics_comparison,
        "communities": {
            "hybrid": hybrid_comms,
            "cpm": cpm_comms,
            "leiden": leiden_comms,
            "bigclam": bigclam_comms,
            "ground_truth": ground_truth
        },
        "bridges": bridges,
        "explanations": explanations,
        "recommendation_eval": rec_eval,
        "cytoscape": cytoscape_elements,
        "node_list": node_list,
        "total_pipeline_time_sec": round(time.time() - t_start, 4)
    }

    # Save to disk
    if save_results:
        out_dir = get_results_dir()
        json_path = os.path.join(out_dir, f"ego_{ego_id}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results_payload, f, indent=2)

        # Save numpy matrix F
        npz_path = os.path.join(out_dir, f"ego_{ego_id}_membership.npz")
        np.savez_compressed(npz_path, F=F_np, node_list=node_list)

    return results_payload

**`backend/scripts/ablation.py`**: Ablation and tuning sweep

In [ ]:
%%writefile backend/scripts/ablation.py
"""
Ablation and tuning sweep for CircleSense (single run per setting, fixed seed).

Part 1: one-factor-at-a-time ablation around the default configuration.
Part 2: small grid search to pick a better configuration.

Only the hybrid model is retrained per setting; seeds and graph tensors are
cached per ego-network. Results are averaged over all ego-networks and written
to backend/experiments/ablation_results.md and ablation_results.csv.

Usage: python backend/scripts/ablation.py [--quick]
"""
import csv
import itertools
import os
import sys
import time

curr = os.path.abspath(os.path.dirname(__file__))
sys.path.insert(0, os.path.abspath(os.path.join(curr, "..")))

import numpy as np
import torch

from app.data.loader import list_egos, load_ego, to_graph_tensors
from app.seeding.cpm import k_clique_seeds
from app.seeding.weak_clique import weak_clique_seeds
from app.seeding.merge import merge_seeds, build_seed_matrix
from app.model.train import train_model, assign_communities
from app.baselines.run import run_leiden_louvain, run_gnn_no_seeds
from app.evaluation.metrics import evaluate_all

SEED = 42
DEFAULT = {
    "k": 4, "seeding": "cpm", "encoder_type": "sage", "structure_loss": "bp",
    "lambda_seed": 1.0, "tau": 0.5, "epochs": 150, "hidden_dim": 64, "lr": 0.01,
}
GAT_MAX_NODES = 800  # GAT here is O(N^2) memory; skip the largest networks
OUT_DIR = os.path.abspath(os.path.join(curr, "..", "experiments"))

_data_cache = {}
_seed_cache = {}


def get_data(ego):
    if ego not in _data_cache:
        G, node_list, node_to_idx, X, truth, feat_names = load_ego(ego)
        _data_cache[ego] = (G, node_list, node_to_idx, X, truth)
    return _data_cache[ego]


def get_seeds(ego, seeding, k):
    key = (ego, seeding, k)
    if key not in _seed_cache:
        G, node_list, node_to_idx, X, truth = get_data(ego)
        raw = weak_clique_seeds(G) if seeding == "weak_clique" else k_clique_seeds(G, k=k)
        merged = merge_seeds(raw, jaccard_threshold=0.6)
        K = max(3, min(len(truth) if truth else 10, len(merged), 20))
        S, _ = build_seed_matrix(merged, node_to_idx, K=K)
        _seed_cache[key] = (S, K)
    return _seed_cache[key]


def mean_metrics(rows):
    keys = ["onmi", "avg_f1", "omega", "bridge_f1"]
    out = {k: round(float(np.mean([r[k] for r in rows])), 4) for k in keys}
    out["circles"] = round(float(np.mean([r["circles"] for r in rows])), 1)
    out["time_s"] = round(float(np.mean([r["time_s"] for r in rows])), 2)
    return out


def run_hybrid(cfg, taus):
    """Train once per ego, evaluate at each tau. Returns {tau: mean metrics}."""
    per_tau = {t: [] for t in taus}
    for ego in list_egos():
        G, node_list, node_to_idx, X, truth = get_data(ego)
        if cfg["encoder_type"] == "gat" and len(node_list) > GAT_MAX_NODES:
            continue
        S, K = get_seeds(ego, cfg["seeding"], cfg["k"])
        _, adj_dense, adj_norm, X_t, S_t = to_graph_tensors(G, X, S)
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        t0 = time.time()
        F_np, _ = train_model(X_t, S_t, adj_norm, adj_dense, cfg)
        elapsed = time.time() - t0
        for t in taus:
            comms = assign_communities(F_np, G, node_list, tau=t)
            m = evaluate_all(comms, truth, node_list)
            m["circles"] = len(comms)
            m["time_s"] = elapsed
            per_tau[t].append(m)
    return {t: mean_metrics(r) for t, r in per_tau.items() if r}


def run_no_seeds(cfg, tau):
    rows = []
    for ego in list_egos():
        G, node_list, node_to_idx, X, truth = get_data(ego)
        _, K = get_seeds(ego, "cpm", cfg["k"])
        _, adj_dense, adj_norm, X_t, _ = to_graph_tensors(G, X, None)
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        t0 = time.time()
        comms, _ = run_gnn_no_seeds(X_t, adj_norm, adj_dense, G, node_list, K=K, tau=tau)
        m = evaluate_all(comms, truth, node_list)
        m["circles"] = len(comms)
        m["time_s"] = time.time() - t0
        rows.append(m)
    return mean_metrics(rows)


def run_louvain():
    rows = []
    for ego in list_egos():
        G, node_list, _, _, truth = get_data(ego)
        t0 = time.time()
        comms, _ = run_leiden_louvain(G)
        m = evaluate_all(comms, truth, node_list)
        m["circles"] = len(comms)
        m["time_s"] = time.time() - t0
        rows.append(m)
    return mean_metrics(rows)


def main():
    quick = "--quick" in sys.argv
    os.makedirs(OUT_DIR, exist_ok=True)
    results = []  # (section, label, metrics)

    def record(section, label, m):
        results.append((section, label, m))
        print(f"  {section:<10} {label:<34} ONMI {m['onmi']:.3f}  F1 {m['avg_f1']:.3f}  "
              f"Omega {m['omega']:.3f}  bridgeF1 {m['bridge_f1']:.3f}  circles {m['circles']}", flush=True)

    t_all = time.time()
    print("Reference baselines")
    record("reference", "Louvain (disjoint)", run_louvain())
    record("reference", "GNN without seeds", run_no_seeds(DEFAULT, DEFAULT["tau"]))

    print("Part 1: ablation (one factor changed from default)")
    record("default", "default", run_hybrid(DEFAULT, [DEFAULT["tau"]])[DEFAULT["tau"]])

    variants = [
        ("seeding", "weak-clique seeds", {"seeding": "weak_clique"}),
        ("k", "k = 3", {"k": 3}),
        ("k", "k = 5", {"k": 5}),
        ("loss", "soft modularity loss", {"structure_loss": "modularity"}),
        ("seed loss", "lambda_seed = 0 (no seed loss)", {"lambda_seed": 0.0}),
        ("seed loss", "lambda_seed = 0.5", {"lambda_seed": 0.5}),
        ("seed loss", "lambda_seed = 2", {"lambda_seed": 2.0}),
        ("seed loss", "lambda_seed = 5", {"lambda_seed": 5.0}),
        ("epochs", "epochs = 50", {"epochs": 50}),
        ("epochs", "epochs = 300", {"epochs": 300}),
        ("encoder", "GAT encoder (N <= 800)", {"encoder_type": "gat"}),
    ]
    if quick:
        variants = variants[:3]
    for section, label, change in variants:
        cfg = {**DEFAULT, **change}
        record(section, label, run_hybrid(cfg, [cfg["tau"]])[cfg["tau"]])

    print("Part 1b: threshold tau (default model, no retraining)")
    taus = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
    for t, m in run_hybrid(DEFAULT, taus).items():
        record("tau", f"tau = {t}", m)

    print("Part 2: grid search")
    grid = {
        "lambda_seed": [0.5, 1.0, 2.0],
        "epochs": [100, 150, 300],
        "k": [3, 4],
        "seeding": ["cpm", "weak_clique"],
    }
    if quick:
        grid = {"lambda_seed": [1.0], "epochs": [150], "k": [4], "seeding": ["cpm"]}
    grid_taus = [0.2, 0.3, 0.5, 0.7]
    best = None
    for vals in itertools.product(*grid.values()):
        change = dict(zip(grid.keys(), vals))
        if change["seeding"] == "weak_clique" and change["k"] != 4:
            continue  # k is unused for weak-clique seeds
        cfg = {**DEFAULT, **change}
        for t, m in run_hybrid(cfg, grid_taus).items():
            label = (f"{change['seeding']} k={change['k']} lam={change['lambda_seed']} "
                     f"ep={change['epochs']} tau={t}")
            results.append(("grid", label, m))
            score = m["onmi"] + m["avg_f1"] + m["omega"]
            if best is None or score > best[0]:
                best = (score, label, m)
    print(f"  best grid setting (sum of ONMI+F1+Omega): {best[1]}")
    print(f"    {best[2]}")

    # Write outputs
    cols = ["section", "setting", "onmi", "avg_f1", "omega", "bridge_f1", "circles", "time_s"]
    with open(os.path.join(OUT_DIR, "ablation_results.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for sec, label, m in results:
            w.writerow([sec, label, m["onmi"], m["avg_f1"], m["omega"], m["bridge_f1"], m["circles"], m["time_s"]])

    def table(rows):
        lines = ["| Setting | ONMI | Avg F1 | Omega | Bridge F1 | Circles | Train s |",
                 "|---|---|---|---|---|---|---|"]
        for _, label, m in rows:
            lines.append(f"| {label} | {m['onmi']} | {m['avg_f1']} | {m['omega']} | "
                         f"{m['bridge_f1']} | {m['circles']} | {m['time_s']} |")
        return "\n".join(lines)

    grid_rows = sorted([r for r in results if r[0] == "grid"],
                       key=lambda r: -(r[2]["onmi"] + r[2]["avg_f1"] + r[2]["omega"]))[:10]
    md = [
        "# Ablation and tuning results",
        "",
        "Mean over all ego-networks, one run per setting (seed 42). Differences of about 0.01-0.02 "
        "are within run-to-run noise and should not be over-interpreted.",
        "",
        "## Reference and ablation (one factor changed from default)",
        "",
        table([r for r in results if r[0] in ("reference", "default", "seeding", "k", "loss",
                                              "seed loss", "epochs", "encoder")]),
        "",
        "## Threshold tau (default model)",
        "",
        table([r for r in results if r[0] == "tau"]),
        "",
        "## Top 10 grid-search settings (ranked by ONMI + F1 + Omega)",
        "",
        table(grid_rows),
        "",
        f"Total runtime: {round(time.time() - t_all)} s",
    ]
    with open(os.path.join(OUT_DIR, "ablation_results.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    print(f"Wrote results to {OUT_DIR} in {round(time.time() - t_all)} s")


if __name__ == "__main__":
    main()

## 3. Imports and reproducibility

In [ ]:
import sys, time, json, random
import numpy as np, pandas as pd, torch, networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, os.path.join(WORK, "backend"))

from app.data.loader import list_egos, load_ego, to_graph_tensors
from app.seeding.cpm import k_clique_seeds
from app.seeding.weak_clique import weak_clique_seeds
from app.seeding.merge import merge_seeds, build_seed_matrix
from app.model.train import train_model, assign_communities
from app.evaluation.metrics import evaluate_all
from app.analysis.bridges import find_bridge_friends
from app.analysis.recommend import suggest_friends, evaluate_suggestions
from app.analysis.explain import explain_circle
from app.pipeline import run_pipeline

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed()
EGOS = list_egos()
print("Ego-networks:", EGOS)

## 4. Walkthrough on one ego-network
Pipeline: `ego-network → clique seeds → merge → seed matrix S → GNN on [X | S] → soft membership F → threshold τ → circles`

In [ ]:
EGO = 0
G, node_list, node_to_idx, X, truth, feat_names = load_ego(EGO)
print(f"Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}  Features: {X.shape[1]}  Ground-truth circles: {len(truth)}")
print("Density:", round(nx.density(G), 4), "| avg clustering:", round(nx.average_clustering(G), 3))

In [ ]:
# Step 1: clique seeds (weak-clique default vs plain k-clique percolation)
t0 = time.time(); weak = weak_clique_seeds(G); t_weak = time.time() - t0
t0 = time.time(); cpm = k_clique_seeds(G, k=4); t_cpm = time.time() - t0
print(f"weak-clique seeds: {len(weak)} ({t_weak:.2f}s) | 4-clique CPM seeds: {len(cpm)} ({t_cpm:.2f}s)")

# Step 2: merge near-duplicate seeds (Jaccard > 0.6) and build seed matrix S (N x K)
merged = merge_seeds(weak, jaccard_threshold=0.6)
K = max(3, min(len(truth), len(merged), 20))
S_np, top_seeds = build_seed_matrix(merged, node_to_idx, K=K)
print(f"after merge: {len(merged)} seeds -> K={K}; S shape {S_np.shape}; nodes covered by a seed: {int((S_np.sum(1) > 0).sum())}/{len(node_list)}")

In [ ]:
# Step 3: train the GNN on [X | S] with L = L_structure + lambda * L_seed
cfg = dict(k=4, seeding="weak_clique", encoder_type="sage", hidden_dim=64, lr=0.01, epochs=300,
           structure_loss="bp", lambda_seed=1.0, tau=0.5)
set_seed()
edge_index, adj_dense, adj_norm, X_t, S_t = to_graph_tensors(G, X, S_np)
F_np, meta = train_model(X_t, S_t, adj_norm, adj_dense, cfg)
print("train meta:", {k: (v if not isinstance(v, (list, np.ndarray)) else f"len {len(v)}") for k, v in meta.items()})
for key in ("loss_history", "losses", "history"):
    if key in meta and hasattr(meta[key], "__len__"):
        plt.figure(figsize=(5, 3)); plt.plot(meta[key]); plt.xlabel("epoch"); plt.ylabel("loss")
        plt.title("Training loss"); plt.tight_layout(); plt.show(); break

In [ ]:
# Step 4: soft membership F -> circles (node joins circle c if F[u,c] >= tau)
circles = assign_communities(F_np, G, node_list, tau=cfg["tau"])
print(f"{len(circles)} circles detected; sizes: {sorted((len(v) for v in circles.values()), reverse=True)}")
print("Metrics vs ground truth:", evaluate_all(circles, truth, node_list))

## 5. Batch run: all 10 ego-networks, Hybrid vs baselines

In [ ]:
rows, results = [], {}
t_all = time.time()
for ego in EGOS:
    set_seed()
    r = run_pipeline(ego, config=None, save_results=True)
    results[ego] = r
    gs = r["graph_summary"]
    for method, mm in r["metrics"].items():
        rows.append({"ego": ego, "nodes": gs["num_nodes"], "edges": gs["num_edges"], "method": method,
                     "ONMI": mm["onmi"], "AvgF1": mm["avg_f1"], "Omega": mm["omega"], "runtime_s": mm["runtime"]})
    h = r["metrics"]["hybrid"]
    print(f"ego {ego:>4} | N={gs['num_nodes']:>4} | circles={gs['num_detected_circles']:>2} | hybrid ONMI {h['onmi']:.3f} F1 {h['avg_f1']:.3f} Omega {h['omega']:.3f}")
print(f"\nTotal: {time.time() - t_all:.1f}s")
df = pd.DataFrame(rows)

In [ ]:
names = {"hybrid": "Hybrid (weak-clique + GNN)", "cpm": "CPM", "leiden": "Louvain (disjoint)", "bigclam": "BigCLAM", "gnn_no_seeds": "GNN without seeds"}
summary = (df.groupby("method")[["ONMI", "AvgF1", "Omega", "runtime_s"]].mean().round(3)
             .reindex(list(names)).rename(index=names))
display(summary)
summary[["ONMI", "AvgF1", "Omega"]].plot(kind="bar", figsize=(9, 4), rot=20,
    title="Mean accuracy over 10 ego-networks (higher is better)")
plt.tight_layout(); plt.show()

In [ ]:
piv = df[df.method == "hybrid"].set_index("ego")[["nodes", "edges", "ONMI", "AvgF1", "Omega", "runtime_s"]]
display(piv)
wins = {}
for meth in ["cpm", "leiden", "bigclam", "gnn_no_seeds"]:
    other = df[df.method == meth].set_index("ego")
    wins[names[meth]] = {c: int((piv[c] > other[c]).sum()) for c in ["ONMI", "AvgF1", "Omega"]}
display(pd.DataFrame(wins).T.rename_axis("Hybrid beats (of 10 networks)"))

## 6. Applications: bridge friends, friend suggestions, circle explanations

In [ ]:
r = results[EGO]
print("Top bridge friends (belong to several circles):")
for b in r["bridges"][:8]:
    print(f"  friend {b['node_id']:>5} | {b['circle_count']} circles | strength {b['total_strength']:.2f} | "
          + ", ".join(f"{c['circle_name']}({c['score']:.2f})" for c in b["circles"]))

In [ ]:
npz = np.load(os.path.join("backend", "results", f"ego_{EGO}_membership.npz"), allow_pickle=True)
F_saved = npz["F"]
G, node_list, node_to_idx, X, truth, feat_names = load_ego(EGO)
user = r["bridges"][0]["node_id"] if r["bridges"] else node_list[0]
print("Suggestions for friend", user)
for s in suggest_friends(F_saved, G, user, node_list, node_to_idx, top_n=5):
    why = ", ".join(f"{c['circle_name']}({c['affinity_contribution']:.2f})" for c in s["shared_circles"][:3])
    print(f"  -> {s['target_node']:>5} | affinity {s['affinity_score']:.2f} | common nbrs {s['common_neighbors_count']} | via {why}")

# Quantitative check: hide 10% of edges; does a hidden friend appear in the top suggestions?
ev = pd.DataFrame({e: results[e]["recommendation_eval"] for e in EGOS}).T
display(ev)
print("Mean hit-rate: hybrid %.2f vs common-neighbours %.2f" % (ev.hybrid_hit_rate.mean(), ev.cn_hit_rate.mean()))

In [ ]:
for ck, ex in list(r["explanations"].items())[:3]:
    print(ck, json.dumps(ex, indent=1, default=str)[:1000])
    print("-" * 60)

## 7. Visualisations

In [ ]:
circ = r["communities"]["hybrid"]
dominant = {}
for ci, (name, mem) in enumerate(circ.items()):
    for n in mem:
        dominant.setdefault(n, ci)
bridge_ids = {b["node_id"] for b in r["bridges"]}
cmap = plt.get_cmap("tab20")
pos = nx.spring_layout(G, seed=SEED, k=0.15)
plt.figure(figsize=(9, 8))
nx.draw_networkx_edges(G, pos, alpha=0.08, width=0.4)
nx.draw_networkx_nodes(G, pos, node_color=[cmap(dominant.get(n, 0) % 20) for n in G.nodes()],
                       node_size=[60 if n in bridge_ids else 22 for n in G.nodes()],
                       edgecolors=["red" if n in bridge_ids else "none" for n in G.nodes()], linewidths=1.2)
plt.title(f"Ego {EGO}: hybrid circles (colour) and bridge friends (red outline)"); plt.axis("off"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
order = np.argsort(F_saved.argmax(1))
ax[0].imshow(F_saved[order].T, aspect="auto", cmap="viridis"); ax[0].set_title("Soft membership F (circles x friends)")
ax[0].set_xlabel("friends (sorted)"); ax[0].set_ylabel("circle")
names_c = list(circ)
ov = np.array([[len(set(circ[a]) & set(circ[b])) for b in names_c] for a in names_c])
im = ax[1].imshow(ov, cmap="magma"); ax[1].set_title("Circle overlap (shared friends)"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for meth in ["hybrid", "cpm", "leiden"]:
    d = df[df.method == meth]; ax[0].scatter(d.nodes, d.AvgF1, label=names[meth])
ax[0].set_xlabel("nodes"); ax[0].set_ylabel("Avg F1"); ax[0].legend(); ax[0].set_title("Avg F1 vs network size")
summary["runtime_s"].plot(kind="barh", ax=ax[1], title="Mean runtime (s)"); plt.tight_layout(); plt.show()

## 8. Optional: ablation and tuning sweep
Re-trains the hybrid model across settings (seeding, k, structure loss, seed-loss weight, epochs, τ, encoder) plus a small grid.
Set `RUN_ABLATION = True`. `QUICK = True` runs a reduced sweep; `QUICK = False` is the full ~15-minute study.

In [ ]:
RUN_ABLATION = False   # set True to run
QUICK = True           # True: reduced sweep, False: full study (~15 min)

if RUN_ABLATION:
    args = [sys.executable, "backend/scripts/ablation.py"] + (["--quick"] if QUICK else [])
    subprocess.run(args, check=True)
    print(open("backend/experiments/ablation_results.md", encoding="utf-8").read())
    display(pd.read_csv("backend/experiments/ablation_results.csv"))
else:
    print("Skipped. Set RUN_ABLATION = True to run the ablation study.")

## 9. Save outputs
Zips the per-network results and summary tables for download from Colab.

In [ ]:
import shutil
summary.to_csv("benchmark_summary.csv"); df.to_csv("benchmark_per_ego.csv", index=False)
shutil.make_archive("circlesense_results", "zip", "backend/results")
print("Wrote benchmark_summary.csv, benchmark_per_ego.csv, circlesense_results.zip in", os.getcwd())
if os.path.isdir("/content"):
    from google.colab import files
    files.download("circlesense_results.zip")

## 10. Interpretation
* **Hybrid** gives the best average F1 and clearly beats plain CPM on every accuracy metric; seeds help a little over the seed-free GNN.
* **Louvain** is a strong baseline on ONMI/Omega on these small graphs, but assigns each friend to one circle, so it cannot find bridge friends.
* Circle-based friend suggestions trail common-neighbours on raw hit-rate but explain *why* each friend is suggested.
* Limitations: single seeded runs, small graphs (dense N×N matrices), simplified baselines, anonymised features.

*Reference: McAuley & Leskovec, "Learning to Discover Social Circles in Ego Networks", NIPS 2012.*